In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
import os
import torch
import torchvision.models
from torch import nn
from torch.utils.data import Dataset as TorchDataset, DataLoader
import torch.optim as TorchOptimizers
import torchvision.transforms.v2 as T
from torchvision.models import MobileNet_V3_Large_Weights, ResNet50_Weights
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large, deeplabv3_resnet50
from torchinfo import summary as torch_summary
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
import math
import time
import wandb
from sklearn.model_selection import train_test_split, GroupShuffleSplit
import json
from typing import Callable
import albumentations
import segmentation_models_pytorch as smp

### Configuration setup

In [ ]:
@dataclass
class Config:
    train_csv_filepath: str
    train_split_csv_filepath: str
    val_split_csv_filepath: str
    test_csv_filepath: str
    submission_filepath: str
    images_root_folder: str
    image_masks_root_folder: str
    training_output_folder: str
    saved_weights_filepath: str
    device: str

    # noinspection PyAttributeOutsideInit
    def init(self, training):
        self.training = training

        if self.training:
            os.makedirs(self.training_output_folder, exist_ok=True)

        self.num_classes = len(CLASS_COLORS)

        self.batch_size = 8
        self.gradient_accumulation_steps = 4
        # effective batch size = batch_size * gradient_accumulation_steps

        self.starting_learning_rate = 4e-4
        self.max_epochs = 30
        self.patience = 3
        self.seed = 1234
        self.num_workers = 4 if self.device == 'cuda' else 0
        self.pin_memory = self.num_workers > 0
        self.image_width = 512
        self.image_height = 512
        self.image_dims = (self.image_width, self.image_height)

        # For mixed precision training, greatly reduces VRAM usage
        self.use_amp = self.device == 'cuda'

        self.imagenet_mean_cpu_tensor = torch.tensor(imagenet_mean_array)
        self.imagenet_std_cpu_tensor = torch.tensor(imagenet_std_array)
        self.channelwise_imagenet_mean_cpu_tensor = self.imagenet_mean_cpu_tensor.view(3, 1, 1)
        self.channelwise_imagenet_std_cpu_tensor = self.imagenet_std_cpu_tensor.view(3, 1, 1)
        self.imagenet_mean_gpu_tensor = gpu_tensor(imagenet_mean_array)
        self.imagenet_std_gpu_tensor = gpu_tensor(imagenet_std_array)
        self.channelwise_imagenet_mean_gpu_tensor = self.imagenet_mean_gpu_tensor.view(3, 1, 1)
        self.channelwise_imagenet_std_gpu_tensor = self.imagenet_std_gpu_tensor.view(3, 1, 1)

        self.image_transforms = T.Compose([
            T.ToImage(),
            T.Resize(self.image_dims, interpolation=T.InterpolationMode.BILINEAR),
            T.ToDtype(torch.float32, scale=True),
            T.Normalize(self.imagenet_mean_cpu_tensor, self.imagenet_std_cpu_tensor),
        ])

config: Config = None
""" Set to environment-relevant config before training/inference """;

In [ ]:
local_config = Config(
    train_csv_filepath='data/train.csv',
    train_split_csv_filepath='data/train_split.csv',
    val_split_csv_filepath='data/val_split.csv',
    test_csv_filepath='data/test.csv',
    submission_filepath='data_gen/submission.csv',
    images_root_folder='data/images/',
    image_masks_root_folder='data/masks/',
    training_output_folder='data_gen/training_output/',
    saved_weights_filepath='data_gen/training_output/best_model_weights.pth',
    device='cpu',
)
kaggle_config = Config(
    train_csv_filepath='/kaggle/input/opencv-pytorch-segmentation-project-round2/train.csv',
    train_split_csv_filepath='/kaggle/input/drone-image-segmentation-trainval-splits/train_split.csv',
    val_split_csv_filepath='/kaggle/input/drone-image-segmentation-trainval-splits/val_split.csv',
    test_csv_filepath='/kaggle/input/opencv-pytorch-segmentation-project-round2/test.csv',
    images_root_folder='/kaggle/input/opencv-pytorch-segmentation-project-round2/imgs/imgs/',
    image_masks_root_folder='/kaggle/input/opencv-pytorch-segmentation-project-round2/masks/masks/',
    submission_filepath='/kaggle/working/submission.csv',
    training_output_folder='/kaggle/working/training_output/',
    saved_weights_filepath='/kaggle/input/deeplabv3/pytorch/drone_image_fine_tune/1/training_output_best_model_weights.pth',
    device='cuda',
)

### Utils

In [ ]:
imagenet_mean_array = np.array([0.485, 0.456, 0.406], dtype=np.float32)
imagenet_std_array = np.array([0.229, 0.224, 0.225], dtype=np.float32)

CLASS_COLORS = np.array([
    [  0,   0,   0],  # 0: Black (Background)
    [192, 128, 128],  # 1: Pink (Person)
    [  0, 128,   0],  # 2: Green (Bike)
    [128, 128, 128],  # 3: Gray (Car)
    [128,   0,   0],  # 4: Maroon (Drone)
    [  0,   0, 128],  # 5: Navy (Boat)
    [192,  64, 128],  # 6: Purple (Animal)
    [192,   0,   0],  # 7: Red (Obstacle)
    [192, 128,   0],  # 8: Orange (Construction)
    [  0,  64,   0],  # 9: Dark Green (Vegetation)
    [128, 128,   0],  # 10: Olive (Road)
    [  0, 128, 128],  # 11: Teal (Sky)
], dtype=np.uint8)

def gpu_tensor(numpy_array):
    return torch.tensor(numpy_array, device=config.device)

def gpu_image_tensor_to_numpy_array(image_tensor):
    image = denormalize(image_tensor, config.channelwise_imagenet_mean_gpu_tensor, config.channelwise_imagenet_std_gpu_tensor)
    image = torch.clamp(image, 0, 1)
    image = image.permute(1, 2, 0).cpu().numpy()
    return (image * 255).astype(np.uint8)

def gpu_mask_tensor_to_colored_mask_numpy_array(mask_tensor):
    mask = mask_tensor.cpu().numpy()
    mask = np.clip(mask, 0, config.num_classes - 1).astype(np.int32)
    return CLASS_COLORS[mask]

def visualize_image(image_tensor):
    """ Input tensor should be on gpu """
    image = denormalize(image_tensor, config.channelwise_imagenet_mean_gpu_tensor, config.channelwise_imagenet_std_gpu_tensor)
    image = torch.clamp(image, 0, 1)
    image = image.permute(1, 2, 0).cpu().numpy()
    image = (image * 255).astype('uint8')
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    plt.close()

def visualize_mask(mask_tensor):
    """ mask_tensor: gpu image tensor with pixel intensity indicating class (0-11) """
    colored_mask = gpu_mask_tensor_to_colored_mask_numpy_array(mask_tensor)
    plt.imshow(colored_mask)
    plt.axis('off')
    plt.show()
    plt.close()

def visualize_mask_overlayed_over_image(image_tensor, mask_tensor, alpha=0.5):
    """ image_tensor: gpu image tensor, mask_tensor: gpu image tensor with pixel intensity indicating class (0-11) """
    image_array = gpu_image_tensor_to_numpy_array(image_tensor)
    colored_mask = gpu_mask_tensor_to_colored_mask_numpy_array(mask_tensor)
    blended = (alpha * colored_mask + (1 - alpha) * image_array).astype(np.uint8)
    plt.imshow(blended)
    plt.axis('off')
    plt.show()
    plt.close()

def normalize(tensor, mean, std):
    return (tensor - mean) / std

def denormalize(tensor, mean, std):
    return tensor * std + mean

### Dataset

In [ ]:
@dataclass
class ImageSegmentationDataset(TorchDataset):
    image_ids: np.ndarray
    image_transforms: Callable

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image = Image.open(f'{config.images_root_folder}{image_id}.jpg')
        mask = Image.open(f'{config.image_masks_root_folder}{image_id}.png')
        # Resize mask using nearest-neighbor to preserve class indices
        mask = mask.resize(config.image_dims, resample=Image.Resampling.NEAREST)
        transformed_image = self.image_transforms(image)
        # Convert mask to tensor directly - no normalization
        # Mask contains class indices 0-11, shape becomes (H, W)
        mask_array = np.array(mask)
        # Set to long (int64) format for crossentropy loss calculation
        transformed_mask = torch.from_numpy(mask_array).long()

        return transformed_image, transformed_mask

### Model

In [ ]:
class CustomDeepLabV3(nn.Module):
    def __init__(self):
        super().__init__()
        # Prevent downloads during inference (relevant for kaggle competitions)
        weights_backbone = ResNet50_Weights.DEFAULT if config.training else None
        # Otherwise, initialize backbone with imagenet weights
        self.model = deeplabv3_resnet50(
            weights=None,
            weights_backbone=weights_backbone,
            num_classes=config.num_classes,
            aux_loss=True,
        )

        for param in self.model.backbone.parameters():
            param.requires_grad = False

        self.model.to(config.device)
        # Freezing batchnorm layers to try to improve generalization
        self._freeze_backbone_bn()

    def unfreeze_layer4(self):
        """ Unfreeze layer4 of the backbone for fine-tuning. """
        for param in self.model.backbone.layer4.parameters():
            param.requires_grad = True

    def _freeze_backbone_bn(self):
        """ Put backbone BN layers into eval mode and freeze their affine parameters. """
        for m in self.model.backbone.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
                # Freeze gamma/beta as well
                if m.affine:
                    m.weight.requires_grad_(False)
                    m.bias.requires_grad_(False)

    def train(self, mode = True):
        """ Override default train() so backbone BN never leaves eval mode. """
        super().train(mode)
        if mode:
            self._freeze_backbone_bn()
        return self

    def forward(self, x):
        return self.model(x)

### Dice Score Helper Functions

In [ ]:
def intersection_and_union_for_batch(pred_masks, true_masks, num_classes):
    """Compute per-class intersection and union for a batch.

    Args:
        pred_masks: (B, C, H, W) raw model output logits
        true_masks: (B, H, W) ground truth class indices
        num_classes: number of classes

    Returns:
        intersection: (num_classes,) tensor
        union: (num_classes,) tensor
    """
    pred_classes = pred_masks.argmax(dim=1)

    intersection = torch.zeros(num_classes, device=pred_masks.device)
    union = torch.zeros(num_classes, device=pred_masks.device)

    for c in range(num_classes):
        pred_c = (pred_classes == c)
        true_c = (true_masks == c)
        intersection[c] = (pred_c & true_c).sum()
        union[c] = pred_c.sum() + true_c.sum()

    return intersection, union

def compute_dice_score(total_intersection, total_union):
    """Compute mean Dice score from accumulated intersection/union values."""
    valid = total_union > 0
    dice = torch.zeros_like(total_intersection)
    dice[valid] = (2 * total_intersection[valid]) / total_union[valid]
    return dice[valid].mean().item()

### Training: One Full Epoch

In [ ]:
def train_one_epoch(epoch_number, start_time, model, loader, optimizer, loss_function, scaler):
    model.train()

    running_loss = 0.0

    # Running sums for incremental Dice computation
    total_intersection = torch.zeros(config.num_classes, device=config.device)
    total_union = torch.zeros(config.num_classes, device=config.device)

    num_batches = math.ceil(len(loader.dataset) / config.batch_size)
    accumulation_steps = config.gradient_accumulation_steps

    optimizer.zero_grad()

    for batch_number, (x, y) in enumerate(loader):
        print(f't={time.time() - start_time:.2f}: Loading training batch {batch_number + 1}/{num_batches}')

        x = x.to(config.device, non_blocking=True)
        y = y.to(config.device, non_blocking=True)

        if batch_number == 0:
            allocated = torch.cuda.memory_allocated(config.device) / 1024**3
            reserved = torch.cuda.memory_reserved(config.device) / 1024**3
            print(f'Memory allocated={allocated:.2f} GiB, reserved={reserved:.2f} GiB')
            print(f'First image with overlayed mask (unique values: {torch.unique(y[0]).tolist()}):')
            visualize_mask_overlayed_over_image(x[0], y[0])

        # Mixed precision forward pass
        with torch.amp.autocast('cuda', enabled=config.use_amp):
            output = model(x)
            preds = output['out'] if isinstance(output, dict) else output
            loss = loss_function(preds, y)
            # Scale loss for gradient accumulation
            loss = loss / accumulation_steps

        # Mixed precision backward pass
        scaler.scale(loss).backward()

        # Update weights every accumulation_steps batches
        if (batch_number + 1) % accumulation_steps == 0 or (batch_number + 1) == num_batches:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * accumulation_steps * x.size(0)

        # Compute batch Dice incrementally (no accumulation)
        with torch.no_grad():
            batch_inter, batch_union = intersection_and_union_for_batch(preds, y, config.num_classes)
            total_intersection += batch_inter
            total_union += batch_union

        # Clean up to free memory
        del output, preds, loss, x, y

    # Clear GPU cache at end of epoch
    if config.device == 'cuda':
        torch.cuda.empty_cache()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_score = compute_dice_score(total_intersection, total_union)

    return epoch_loss, epoch_score

### Validation: One Full Epoch

In [ ]:
@torch.no_grad()
def validate_one_epoch(epoch_number, start_time, model, loader, loss_function):
    model.eval()
    running_loss = 0.0

    # Running sums for incremental Dice computation
    total_intersection = torch.zeros(config.num_classes, device=config.device)
    total_union = torch.zeros(config.num_classes, device=config.device)

    num_batches = math.ceil(len(loader.dataset) / config.batch_size)
    for batch_number, (x, y) in enumerate(loader):
        print(f't={time.time() - start_time:.2f}: Loading validation batch {batch_number + 1}/{num_batches}')

        x = x.to(config.device, non_blocking=True)
        y = y.to(config.device, non_blocking=True)

        if batch_number == 0:
            print('First image:')
            visualize_image(x[0])
            print(f'First mask (unique values: {torch.unique(y[0]).tolist()}):')
            visualize_mask(y[0])

        # Mixed precision inference
        with torch.amp.autocast('cuda', enabled=config.use_amp):
            output = model(x)
            preds = output['out'] if isinstance(output, dict) else output
            loss = loss_function(preds, y)

        running_loss += loss.item() * x.size(0)

        # Compute batch Dice incrementally (no accumulation)
        batch_inter, batch_union = intersection_and_union_for_batch(preds, y, config.num_classes)
        total_intersection += batch_inter
        total_union += batch_union

        # Clean up to free memory
        del output, preds, loss, x, y

    # Clear GPU cache at end of epoch
    if config.device == 'cuda':
        torch.cuda.empty_cache()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_score = compute_dice_score(total_intersection, total_union)

    return epoch_loss, epoch_score

### Training: Main Loop

In [ ]:
def train():
    config.init(training=True)

    start_time = time.time()
    print('t=0: Starting data prep and model loading')

    effective_batch_size = config.batch_size * config.gradient_accumulation_steps
    run = wandb.init(
        project='drone_image_segmentation',
        name=f'run={int(start_time)}',
        config={
            'batch_size': config.batch_size,
            'gradient_accumulation_steps': config.gradient_accumulation_steps,
            'effective_batch_size': effective_batch_size,
            'learning_rate': config.starting_learning_rate,
            'max_epochs': config.max_epochs,
            'seed': config.seed,
            'model': 'deeplabv3_resnet50',
            'optimizer': 'Adam',
            'image_size': config.image_dims,
            'use_amp': config.use_amp,
        },
    )

    train_ids = pd.read_csv(config.train_csv_filepath)['ImageID'].to_numpy()
    val_ids = pd.read_csv(config.val_split_csv_filepath)['ImageID'].to_numpy()

    print(f'Train size: {len(train_ids)}, Val size: {len(val_ids)}')

    model = CustomDeepLabV3()

    wandb.watch(model, log='gradients', log_freq=100)

    train_dataset = ImageSegmentationDataset(train_ids, config.image_transforms)
    val_dataset = ImageSegmentationDataset(val_ids, config.image_transforms)

    def loader(ds, shuffle):
        return DataLoader(ds, shuffle=shuffle, batch_size=config.batch_size, num_workers=config.num_workers, pin_memory=config.pin_memory)

    train_loader = loader(train_dataset, shuffle=True)
    val_loader = loader(val_dataset, shuffle=False)

    # Use CrossEntropyLoss for multi-class segmentation with class indices
    loss_function = nn.CrossEntropyLoss()
    optimizer = TorchOptimizers.Adam(model.parameters(), lr=config.starting_learning_rate)

    # Mixed precision scaler
    scaler = torch.amp.GradScaler('cuda', enabled=config.use_amp)

    best_val_loss = float('inf')
    best_val_loss_epoch = -1
    best_val_score = float('-inf')
    best_val_score_epoch = -1

    history = dict(train_loss=[], val_loss=[], train_score=[], val_score=[], best_val_score_epoch=dict(), best_val_loss_epoch=dict())

    training_start_time = time.time()
    print(f't={training_start_time - start_time:.2f}: Starting training')
    print(f'Batch size: {config.batch_size}, Gradient accumulation: {config.gradient_accumulation_steps}, Effective batch size: {effective_batch_size}')
    print(f'Image size: {config.image_dims}, AMP enabled: {config.use_amp}')
    torch.manual_seed(config.seed)

    best_score_weights_path = config.training_output_folder + 'best_model_weights.pth'
    best_loss_weights_path = config.training_output_folder + 'best_loss_model_weights.pth'

    saved_weights_paths = []

    epochs_since_best = 0

    for epoch in range(1, config.max_epochs + 1):
        epoch_start_time = time.time()
        print(f't={epoch_start_time - start_time:.2f}: Starting epoch {epoch}')

        if epoch == 4:
            layer4_lr = config.starting_learning_rate * 0.25
            print(f't={time.time() - start_time:.2f}: Unfreezing layer4 of resnet50 backbone for fine-tuning; learning rate = 0.25 * config.starting_learning_rate, i.e. 0.25 * {config.starting_learning_rate}, i.e. {layer4_lr}')
            model.unfreeze_layer4()

            # Rebuild optimizer with two parameter groups
            layer4_params = list(model.model.backbone.layer4.parameters())
            layer4_param_ids = {id(p) for p in layer4_params}
            other_params = [p for p in model.parameters() if p.requires_grad and id(p) not in layer4_param_ids]

            optimizer = TorchOptimizers.Adam([
                {'params': other_params, 'lr': config.starting_learning_rate},
                {'params': layer4_params, 'lr': layer4_lr},
            ])

        train_loss, train_score = train_one_epoch(epoch, start_time, model, train_loader, optimizer, loss_function, scaler)
        val_loss, val_score = validate_one_epoch(epoch, start_time, model, val_loader, loss_function)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_score'].append(train_score)
        history['val_score'].append(val_score)

        print(f'================ Epoch {epoch:03d} stats ==================')
        print(f'train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}')
        print(f'train_score: {train_score:.4f}  val_score: {val_score:.4f}')
        print('===================================================')

        wandb.log(
            {
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'train_score': train_score,
                'val_score': val_score,
            }
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_loss_epoch = epoch
            torch.save(model.state_dict(), best_loss_weights_path)

        if val_score > best_val_score:
            best_val_score = val_score
            best_val_score_epoch = epoch
            epochs_since_best = 0
            torch.save(model.state_dict(), best_score_weights_path)
        else:
            epochs_since_best += 1
            if epochs_since_best >= config.patience:
                break

    history['best_val_score_epoch']['epoch'] = best_val_score_epoch
    history['best_val_score_epoch']['val_score'] = best_val_score
    history['best_val_loss_epoch']['epoch'] = best_val_loss_epoch
    history['best_val_loss_epoch']['val_loss'] = best_val_loss

    print()
    print('==================== Results ======================')
    print(f'Best val score epoch: {best_val_score_epoch}')
    print(f'Best val score: {best_val_score:.4f}')
    print(f'Best val loss epoch: {best_val_loss_epoch}')
    print(f'Best val loss: {best_val_loss:.2f}')
    print('===================================================')
    print()

    wandb.run.summary['best_val_score'] = best_val_score
    wandb.run.summary['best_val_score_epoch'] = best_val_score_epoch
    wandb.run.summary['best_val_loss'] = best_val_loss
    wandb.run.summary['best_val_loss_epoch'] = best_val_loss_epoch

    train_score = history['train_score']
    val_score = history['val_score']
    epochs = list(range(1, len(train_score) + 1))

    plt.figure(figsize=(8, 5))

    plt.plot(epochs, train_score, label='train_score', marker='o')
    plt.plot(epochs, val_score, label='val_score', marker='o')

    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Train vs Validation Score per Epoch')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()

    wandb.log({"score_curve": wandb.Image(plt.gcf())})

    plt.show()
    plt.close()

    with open(config.training_output_folder + 'history.json', 'w') as json_file:
        json.dump(history, json_file, indent=4)

    wandb.save(best_score_weights_path)
    wandb.save(best_loss_weights_path)
    wandb.save(config.training_output_folder + 'history.json')

    wandb.finish()

### Inference: Helper Functions

In [ ]:
def load_saved_model() -> nn.Module:
    loaded_model = CustomDeepLabV3()
    saved_model_weights = torch.load(config.saved_weights_filepath, weights_only=True, map_location='cpu')
    loaded_model.load_state_dict(saved_model_weights)
    loaded_model.eval()
    return loaded_model

@torch.no_grad()
def predict_single_image(model, image_id) -> np.ndarray:
    model.eval()
    image = Image.open(f"{config.images_root_folder}{image_id}.jpg")
    transformed_image = config.image_transforms(image)
    input_tensor = transformed_image.unsqueeze(0).to(config.device)

    # Run inference with mixed precision if enabled
    with torch.amp.autocast("cuda", enabled=config.use_amp):
        output = model(input_tensor)
        preds = output["out"] if isinstance(output, dict) else output

    # Get class predictions: (1, C, H, W) -> (1, H, W) -> (H, W)
    pred_mask = preds.argmax(dim=1).squeeze(0)

    return pred_mask.cpu().numpy()

### Validation Set Inference (Sanity Check)

In [ ]:
@torch.no_grad()
def predict_val(num_visualize, exit_early):
    """ For verifying that the saved weights load and predict correctly on the original validation dataset """
    config.init(training=False)

    val_ids = pd.read_csv(config.val_split_csv_filepath)['ImageID'].to_numpy()

    loaded_model = load_saved_model()

    # Create validation dataset and loader
    val_dataset = ImageSegmentationDataset(val_ids, config.image_transforms)
    val_loader = DataLoader(
        val_dataset,
        shuffle=False,
        batch_size=config.batch_size,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
    )

    # Running sums for incremental Dice computation
    total_intersection = torch.zeros(config.num_classes, device=config.device)
    total_union = torch.zeros(config.num_classes, device=config.device)

    num_batches = math.ceil(len(val_dataset) / config.batch_size)
    visualized = 0

    for batch_number, (x, y) in enumerate(val_loader):
        print(f"Processing validation batch {batch_number + 1}/{num_batches}")

        x = x.to(config.device, non_blocking=True)
        y = y.to(config.device, non_blocking=True)

        # Run inference with mixed precision if enabled
        with torch.amp.autocast("cuda", enabled=config.use_amp):
            output = loaded_model(x)
            preds = output["out"] if isinstance(output, dict) else output

        # Compute batch Dice incrementally
        batch_inter, batch_union = intersection_and_union_for_batch(preds, y, config.num_classes)
        total_intersection += batch_inter
        total_union += batch_union

        # Visualize some predictions
        if visualized < num_visualize:
            pred_masks = preds.argmax(dim=1)
            for i in range(min(x.size(0), num_visualize - visualized)):
                print(f"\n--- Sample {visualized + 1} ---")
                print("Input image:")
                visualize_image(x[i])
                print("Ground truth mask:")
                visualize_mask(y[i])
                print("Predicted mask:")
                visualize_mask(pred_masks[i])
                print("Actual mask overlay:")
                visualize_mask_overlayed_over_image(x[i], y[i])
                print("Predict mask overlay:")
                visualize_mask_overlayed_over_image(x[i], pred_masks[i])
                visualized += 1
        elif exit_early:
            break

        del output, preds, x, y

    # Compute final mean Dice score
    mean_dice = compute_dice_score(total_intersection, total_union)

    print(f"\n{'=' * 50}")
    print(f"Validation Mean Dice Score: {mean_dice:.4f}")
    print(f"{'=' * 50}")

    return mean_dice

### Inference: Test Set

In [ ]:
def rle_encode(mask):
    """
    Run-length encode a binary mask.

    Args:
        mask: 2D numpy array (H, W) with binary values (0 or 1)

    Returns:
        RLE string with format "start1 length1 start2 length2 ..."
        Pixels are 0-indexed, column-major order (top to bottom, then left to right).
    """
    # Flatten the mask in column-major (Fortran) order
    # This gives top-to-bottom, then left-to-right pixel ordering
    pixels = mask.flatten(order='F')

    # Pad with zeros at start and end to detect transitions
    pixels = np.concatenate([[0], pixels, [0]])

    # Find where values change
    runs = np.where(pixels[1:] != pixels[:-1])[0]

    # Get start positions and lengths
    # runs[0::2] are starts of 1-runs, runs[1::2] are ends
    runs[1::2] -= runs[::2]

    # Positions are already 0-indexed from np.where

    return " ".join(str(x) for x in runs)

@torch.no_grad()
def predict_test(num_visualize, exit_early):
    """
    Generate predictions on the test set and create submission file.

    Predictions are resized to match original image dimensions and
    encoded using Run-Length Encoding (RLE) for each class.

    Args:
        num_visualize: Number of test images to visualize (default: 3)
        exit_early: Stop inference after num_visualize images
    """
    config.init(training=False)

    # Load saved model
    model = load_saved_model()

    # Read test image IDs
    test_df = pd.read_csv(config.test_csv_filepath)
    test_ids = test_df["ImageID"].to_numpy()

    num_images = len(test_ids)
    submission_rows = []
    visualized = 0

    for idx, image_id in enumerate(test_ids):
        if (idx + 1) % 50 == 0 or idx == 0:
            print(f"Processing test image {idx + 1}/{num_images}")

        # Load image and get original dimensions
        image_path = f"{config.images_root_folder}{image_id}.jpg"
        image = Image.open(image_path)
        original_width, original_height = image.size

        # Transform and run inference
        transformed_image = config.image_transforms(image)
        input_tensor = transformed_image.unsqueeze(0).to(config.device)

        with torch.amp.autocast("cuda", enabled=config.use_amp):
            output = model(input_tensor)
            preds = output["out"] if isinstance(output, dict) else output

        # Get class predictions: (1, C, H, W) -> (H, W)
        pred_mask = preds.argmax(dim=1).squeeze(0)

        # Visualize some predictions
        if visualized < num_visualize:
            print(f"\n--- Test Sample {visualized + 1} (ImageID: {image_id}) ---")
            print("Input image:")
            visualize_image(input_tensor.squeeze(0))
            print("Predicted mask:")
            visualize_mask(pred_mask)
            print("Overlay:")
            visualize_mask_overlayed_over_image(input_tensor.squeeze(0), pred_mask)
            visualized += 1
        elif exit_early:
            break

        pred_mask_np = pred_mask.cpu().numpy()

        # Resize prediction to original image dimensions using nearest neighbor
        pred_mask_pil = Image.fromarray(pred_mask_np.astype(np.uint8))
        pred_mask_resized = pred_mask_pil.resize(
            (original_width, original_height), resample=Image.NEAREST
        )
        pred_mask_original = np.array(pred_mask_resized)

        # Generate RLE for each class
        for class_idx in range(config.num_classes):
            binary_mask = (pred_mask_original == class_idx).astype(np.uint8)
            rle = rle_encode(binary_mask)
            submission_rows.append({
                "ImageID": f"{image_id}_{class_idx}",
                "EncodedPixels": rle,
            })

    # Create submission dataframe and save
    submission_df = pd.DataFrame(submission_rows)
    submission_df.to_csv(config.submission_filepath, index=False)

    print(f"\nSubmission saved to: {config.submission_filepath}")
    print(f"Total rows: {len(submission_df)}")

    return submission_df

### Kaggle: WandB login

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# wandb_key = user_secrets.get_secret("wandb_key")
# !wandb login $wandb_key

### Run Training/Inference

In [ ]:
config = local_config # local_config / kaggle_config
train() # train() / predict_val() / predict_test()